In [1]:
from pathlib import Path
import sys
import sympy as sp

# Cho phép import khi mở notebook từ thư mục project hoặc notebooks/
root = Path.cwd()
if not (root / "src").exists() and (root.parent / "src").exists():
    root = root.parent

sys.path.insert(0, str(root / "src" / "lib"))
sys.path.insert(0, str(root / "src"))

from geometry_engine import GeometryEngine

In [ ]:
eng = GeometryEngine()

# Add points
for name in ["A", "B", "C", "D", "P"]:
    eng.add_point(name)

# Set P as origin
eng.value_subs[eng.z_symbol("P")] = sp.Integer(0)
eng.value_subs[eng.zb_symbol("P")] = sp.Integer(0)

# Constraints: A,P,C collinear and B,P,D collinear
eng.add_collinear("A", "P", "C")
eng.add_collinear("B", "P", "D")

# ABCD is cyclic
eng.add_concyclic("A", "B", "C", "D")

# Construct P-Dumpty points for triangles PAB, PBC, PCD, PDA
eng.dumpty_point("P", "A", "B", "X")
eng.dumpty_point("P", "B", "C", "Y")
eng.dumpty_point("P", "C", "D", "Z")
eng.dumpty_point("P", "D", "A", "W")

# Midpoints
eng.midpoint("X", "Z", "M")
eng.midpoint("Y", "W", "N")

(z_A*z_D*(z_B + z_C) + z_B*z_C*(z_A + z_D))/(2*(z_A + z_D)*(z_B + z_C))

In [ ]:
# Check collinearity M, P, N
num, den = eng.constraint_conjugate_free("collinear", ["M", "P", "N"])[0]
num_s = sp.simplify(eng._apply_all(num))
den_s = sp.simplify(eng._apply_all(den))

print("Numerator:")
print(eng.format_expr(num_s, style="text"))
print("\nDenominator:")
print(eng.format_expr(den_s, style="text"))
print("\nCollinear?", sp.simplify(num_s) == 0)

Numerator:
-A**2*B**2*C*conjugate(A)**2*conjugate(B)*conjugate(C)**2 - 2*A**2*B**2*C*conjugate(A)**2*conjugate(B)*conjugate(C)*conjugate(D) - A**2*B**2*C*conjugate(A)**2*conjugate(B)*conjugate(D)**2 - A**2*B**2*C*conjugate(A)**2*conjugate(C)**2*conjugate(D) - A**2*B**2*C*conjugate(A)**2*conjugate(C)*conjugate(D)**2 - A**2*B**2*C*conjugate(A)*conjugate(B)**2*conjugate(C)**2 - 2*A**2*B**2*C*conjugate(A)*conjugate(B)**2*conjugate(C)*conjugate(D) - A**2*B**2*C*conjugate(A)*conjugate(B)**2*conjugate(D)**2 - 2*A**2*B**2*C*conjugate(A)*conjugate(B)*conjugate(C)**2*conjugate(D) - 2*A**2*B**2*C*conjugate(A)*conjugate(B)*conjugate(C)*conjugate(D)**2 - A**2*B**2*C*conjugate(B)**2*conjugate(C)**2*conjugate(D) - A**2*B**2*C*conjugate(B)**2*conjugate(C)*conjugate(D)**2 - A**2*B**2*D*conjugate(A)**2*conjugate(B)*conjugate(C)**2 - 2*A**2*B**2*D*conjugate(A)**2*conjugate(B)*conjugate(C)*conjugate(D) - A**2*B**2*D*conjugate(A)**2*conjugate(B)*conjugate(D)**2 - A**2*B**2*D*conjugate(A)**2*conjugate(C)**2

example of the engine not working this should return true

In [ ]:
from IPython.display import display, Markdown, Math

names = ["A", "B", "C", "D", "P", "X", "Y", "Z", "W", "M", "N"]
latex_summary = eng.point_summary(names, style="latex")

rows = ["| Point | Coordinate |", "|---|---|"]
for name in names:
    rows.append(f"| ${name}$ | ${latex_summary[name]}$ |")
display(Markdown("\n".join(rows)))

for name in names:
    display(Math(f"{name} = {latex_summary[name]}"))

In [ ]:
# Expand M and print (m-p)/conj(m-p)
zM = sp.expand(sp.together(eng._apply_all(eng.z("M"))))
zbM = sp.expand(sp.together(eng._apply_all(eng.zb("M"))))
zN = sp.simplify(eng._apply_all(eng.z("N")))
zbN = sp.simplify(eng._apply_all(eng.zb("N")))

zP = sp.simplify(eng._apply_all(eng.z("P")))
zbP = sp.simplify(eng._apply_all(eng.zb("P")))
ratio_M = sp.simplify(sp.together((zM - zP) / (zbM - zbP)))

print("M (expanded) =", eng.format_expr(zM, style="text"))
print("conj(M) (expanded) =", eng.format_expr(zbM, style="text"))
print("N (simplified) =", eng.format_expr(zN, style="text"))
print("conj(N) (simplified) =", eng.format_expr(zbN, style="text"))
print("\n(m-p)/conj(m-p) =", eng.format_expr(ratio_M, style="text"))

display(Math(r"m = " + eng.format_expr(zM, style="latex")))
display(Math(r"n = " + eng.format_expr(zN, style="latex")))
display(Math(r"\frac{m-p}{\overline{m-p}} = " + eng.format_expr(ratio_M, style="latex")))